# RNAPhaseek — score and design protein-free LLPS RNAs

**One-click Colab for the RNAPhaseek production model.**

RNAPhaseek predicts the probability that an RNA undergoes **protein-free liquid–liquid phase separation (LLPS)**.

**What this notebook does:**
- **Score** any RNA sequence(s) you paste, getting P(LLPS) per sequence.
- **Design** new high-scoring RNAs de novo with a GA or DEN generator.

### Runtime
GPU is recommended but not required. *Runtime → Change runtime type → T4 GPU* for ~5× speed.

## 1. Setup (run once per session — ~3 min)

Installs dependencies, clones the repo, and downloads the model weights from Hugging Face Hub.

In [ ]:
#@title Setup (run once per session, ~1–2 min) { display-mode: "form" }
GITHUB_REPO = "QuercusCode/RNAPhaseek"
HF_REPO_ID  = "quercuscode/rnaphaseek"

import os, sys, pathlib

# --- Python deps (ViennaRNA + project deps; pinned to the kernel's Python) ---
print("[setup] installing Python deps")
!{sys.executable} -m pip install -q viennarna multimolecule "transformers>=4.40" "torch>=2.1" biopython huggingface_hub tqdm scikit-learn matplotlib pandas

import RNA
print(f"[setup] ViennaRNA OK, v={RNA.__version__ if hasattr(RNA,'__version__') else '?'}")

# --- Project code ---
if not pathlib.Path("RNAPhaseek").exists():
    print(f"[setup] cloning {GITHUB_REPO}")
    !git clone -q https://github.com/{GITHUB_REPO}.git RNAPhaseek
%cd RNAPhaseek

# --- Weights from HF Hub ---
from huggingface_hub import hf_hub_download
print(f"[setup] downloading model weights from {HF_REPO_ID}")
MODEL_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="final_model.pt")
NORM_PATH  = hf_hub_download(repo_id=HF_REPO_ID, filename="norm_stats.npz")

# Place them where rnaphaseek.py expects them (so its default paths just work)
os.makedirs("model/production", exist_ok=True)
for src, dst in [(MODEL_PATH, "model/production/final_model.pt"),
                 (NORM_PATH,  "model/production/norm_stats.npz")]:
    if not pathlib.Path(dst).exists():
        os.symlink(src, dst)

print("[setup] OK — proceed to the next cell.")

## 2. Load the scorer (run once)

In [ ]:
import sys
sys.path.insert(0, ".")
from rnaphaseek import RNAPhaseekScorer, read_fasta
import torch, io, csv, numpy as np

scorer = RNAPhaseekScorer(
    model_path="model/production/final_model.pt",
    norm_path="model/production/norm_stats.npz",
    quiet=False,
)
print(f"\nRNAPhaseek ready on {scorer.device}")

## 3. Score your RNA sequences

Paste your sequence(s) in the code cell below (replace the examples). Use standard FASTA format — a `>name` line followed by the sequence on the next line(s). One sequence or many, both work.

> **Length note**: the model has a **1022-nt context window** (it's built on RNA-FM). Longer sequences are scored on their first 1022 nt — a held-out study showed no accuracy loss vs full-length scoring for this task. If you suspect signal lives past nt 1022 in a very long RNA, the full CLI offers a `--long-model mil` mode that reads the whole sequence.

In [ ]:
# ──────────────────────────────────────────────────────────────
# 1. PASTE YOUR SEQUENCES HERE — between the triple quotes.
#    Standard FASTA: ">name" line, then the sequence (RNA: A/C/G/U).
# ──────────────────────────────────────────────────────────────
input_fasta = """\
>example_g4_repeat
GGGAGGGAGGGAGGGUUUUUUUUUUUUUUUUUUUUU
>example_au_rich
UUUAAAUUUAAAUUUAAAUUUAAAUUUAAAUUUAAA
>example_random
ACGUACGUACGUACGUACGUACGUACGUACGUACGU
"""

# Alternative: upload a .fasta file from your computer.
# Uncomment these 3 lines, run the cell, and pick the file in the popup.
# from google.colab import files
# uploaded = files.upload()
# input_fasta = next(iter(uploaded.values())).decode()

# ──────────────────────────────────────────────────────────────
# 2. OPTIONS
# ──────────────────────────────────────────────────────────────
threshold = 0.5       # P(LLPS) cutoff that flips "LLPS" / "no"
save_csv  = True      # download scores.csv at the end

# ──────────────────────────────────────────────────────────────
# 3. SCORE (no editing needed below)
# ──────────────────────────────────────────────────────────────
import pandas as pd

with open("/tmp/input.fasta", "w") as f:
    f.write(input_fasta.strip() + "\n")
recs = read_fasta("/tmp/input.fasta")
seqs = [RNAPhaseekScorer._norm_seq(s) for _, s in recs]
if not seqs:
    raise SystemExit("No sequences parsed. Did you forget the '>' header line?")

print(f"Scoring {len(seqs)} sequence(s) ...")
probs = scorer.score(seqs)

rows = [("id", "length", "GC_percent", "P_LLPS", f"call@{threshold:.2f}")]
for (h, _), s, p in zip(recs, seqs, probs):
    gc = 100 * sum(c in "GC" for c in s) / max(len(s), 1)
    rows.append((h.split()[0], len(s), f"{gc:.1f}", f"{p:.4f}", "LLPS" if p >= threshold else "no"))

df = pd.DataFrame(rows[1:], columns=rows[0])
print()
print(df.to_string(index=False))

if save_csv:
    df.to_csv("scores.csv", index=False)
    from google.colab import files
    files.download("scores.csv")
    print("\nscores.csv downloaded to your machine.")

## 4. Design — GA generator (~30–60 s)

A **genetic algorithm** starts from random RNA sequences and evolves them toward sequences the model thinks will phase-separate: each generation, the best are kept, the rest are mutated and recombined, and the cycle repeats. Fast and reliable — usually under a minute for the defaults.

> **Output**: a FASTA with the top *N* designs, each header tagged with the model's predicted P(LLPS), e.g. `>ga_design_0_P0.987`.
>
> For a **diverse library** (many distinct designs instead of close variants of one motif), use the DEN generator in the next section instead.

In [ ]:
#@title GA design { display-mode: "form" }

#@markdown **Sequence length** (nt) — how long each design will be. Capped at ~1000 by the model's context window.
length = 100 #@param {type:"slider", min:30, max:1000, step:10}

#@markdown **Number of designs** to return — the top *N* scored by predicted P(LLPS).
n_designs = 5 #@param {type:"slider", min:1, max:30, step:1}

#@markdown **Generations** — more = better designs but slower (~1–2 s/gen on a T4 GPU). 25 is a good default; bump to 50+ if best scores are still climbing in the log.
generations = 25 #@param {type:"slider", min:5, max:80, step:5}

#@markdown **Random seed** — fixed = reproducible; change it (any integer) to explore a different set of designs.
seed = 0 #@param {type:"integer"}

#@markdown **Download FASTA** to your computer when done.
download_fasta = True #@param {type:"boolean"}

#@markdown ---
#@markdown ⚠️ Designs are **model-believed candidates**. A high P(LLPS) means the predictor expects them to phase-separate — they still need experimental validation. Designs on low-complexity (e.g. very GC-poor or homopolymer-like) sequences can occasionally be composition artefacts; for a quick sanity check, score a di-shuffled copy of a design and confirm the score drops.

import random, numpy as np

rng = random.Random(seed)
BASES = "ACGU"
POP, ELITE, MUT, TOURN = 64, 10, 0.04, 3
pop = ["".join(rng.choice(BASES) for _ in range(length)) for _ in range(POP)]
cache = {}
for g in range(generations):
    new = [s for s in pop if s not in cache]
    if new:
        scored = scorer.score(new)
        for s, p in zip(new, scored): cache[s] = float(p)
    fit = [cache[s] for s in pop]
    order = sorted(range(POP), key=lambda i: -fit[i])
    if (g + 1) % 5 == 0 or g == 0:
        print(f"  gen {g+1:>2}/{generations}: best={fit[order[0]]:.4f}  mean={np.mean(fit):.4f}")
    elites = [pop[i] for i in order[:ELITE]]; newpop = list(elites)
    def tour():
        c = rng.sample(range(POP), TOURN); return pop[max(c, key=lambda i: fit[i])]
    while len(newpop) < POP:
        x, y = tour(), tour(); i, j = sorted(rng.sample(range(1, length), 2))
        child = x[:i] + y[i:j] + x[j:]
        child = "".join(rng.choice(BASES) if rng.random() < MUT else ch for ch in child)
        newpop.append(child)
    pop = newpop

top = sorted(set(pop), key=lambda s: -cache.get(s, 0))[:n_designs]
print(f"\nTop {n_designs} GA designs (header shows predicted P(LLPS)):")
lines = []
for i, s in enumerate(top):
    h = f">ga_design_{i}_P{cache[s]:.3f}"
    print(f"  {h}\n  {s}")
    lines.append(h); lines.append(s)

if download_fasta:
    open("designs_ga.fasta", "w").write("\n".join(lines) + "\n")
    from google.colab import files; files.download("designs_ga.fasta")
    print("\ndesigns_ga.fasta downloaded.")

## 5. Design — DEN generator (diverse library, ~2–5 min)

A **Deep Exploration Network** trains a small neural network that, instead of returning one motif and variants, outputs a **mutually diverse batch** of high-P(LLPS) sequences (low pairwise identity). Use this when you want a screening library rather than one hit.

> **Slower than the GA** — it has to train the generator first (~2–5 min on a T4 GPU at the defaults).
>
> **Output**: a FASTA with the diverse designs; each header includes the model's predicted P(LLPS).
>
> **Watch the log**: `fitness` should climb and `pairwise_sim` should stay low or fall — that's the model finding high-scoring sequences that are also distinct from each other.

In [ ]:
#@title DEN design { display-mode: "form" }

#@markdown **Sequence length** (nt) for each design. Capped at ~1000 by the model's context window.
length = 200 #@param {type:"slider", min:50, max:1000, step:10}

#@markdown **Library size** — how many mutually distinct designs to return.
n_designs = 10 #@param {type:"slider", min:5, max:50, step:1}

#@markdown **Training steps** — how long to train the generator. More = converges better but slower. 200 is a good default; bump to 400+ if `fitness` is still rising in the log.
steps = 200 #@param {type:"slider", min:50, max:600, step:50}

#@markdown **Random seed** — fixed = reproducible; change it to get a different library.
seed = 0 #@param {type:"integer"}

#@markdown **Download FASTA** to your computer when done.
download_fasta = True #@param {type:"boolean"}

#@markdown ---
#@markdown ⚠️ Designs are **model-believed candidates** — high P(LLPS) means the predictor expects them to phase-separate, but they still need experimental validation.

import subprocess, sys, pathlib
out_fa = f"designs_den_{length}nt.fasta"
subprocess.run([sys.executable, "scripts/generation/den_design_v6.py",
                "--length", str(length), "--n", str(n_designs),
                "--steps", str(steps), "--seed", str(seed), "--out", out_fa],
               check=True)

print(f"\nDesigns written to {out_fa}:")
print(open(out_fa).read())

if download_fasta:
    from google.colab import files; files.download(out_fa)
    print(f"{out_fa} downloaded.")

## 6. Visualize secondary structure (2D, instant)

A quick 2D view of the RNA's base-pairing pattern — the **minimum free energy (MFE) secondary structure** predicted by ViennaRNA. Useful for spotting **G-quadruplexes, stems, kissing loops**, and the structured-vs-disordered character of a sequence — exactly the motifs that drive RNA-self-LLPS.

> A static SVG renders below the form. For a **draggable, zoomable view** (rearrange nodes, alternate layouts), the cell also prints an interactive `forna` URL you can open in a new tab.

In [ ]:
#@title Fold and visualize (2D) { display-mode: "form" }

#@markdown **Sequence** to fold — paste your own or copy one of your top designs.
sequence = "GGGAGGGAGGGAGGGUUUUUUUUUUUUUUUUUUUUU" #@param {type:"string"}

#@markdown ---

import RNA
from IPython.display import SVG, display
from urllib.parse import quote

seq = sequence.upper().replace("T", "U").strip()
if not seq or any(c not in "ACGU" for c in seq):
    raise SystemExit(f"Sequence must contain only A/C/G/U (T accepted, mapped to U). Got: {sequence[:50]!r}")

# Fold with ViennaRNA — MFE structure
fc = RNA.fold_compound(seq)
(structure, mfe) = fc.mfe()

print(f"Length:    {len(seq)} nt")
print(f"MFE:       {mfe:.2f} kcal/mol")
print(f"Sequence:  {seq}")
print(f"Structure: {structure}")
print("           (dot-bracket: '.' = unpaired, matching '(' and ')' = base pair)")

# Inline SVG
RNA.svg_rna_plot(seq, structure, "/tmp/rna_struct.svg")
display(SVG("/tmp/rna_struct.svg"))

# Interactive forna URL (drag nodes, zoom, alternate layouts) — opens in a new tab
forna_fasta = quote(f">user_seq\n{seq}\n{structure}", safe="")
forna_url   = f"http://rna.tbi.univie.ac.at/forna/forna.html?id=fasta&file={forna_fasta}"
print(f"\nInteractive draggable view: {forna_url}")

## 7. Visualize 3D structure (optional, ~1–3 min, GPU required)

Predicts the RNA's 3D atomic structure with **Boltz-2** — an open-source AlphaFold3-equivalent that handles RNA, DNA, proteins, and ligands — then renders it inline with py3Dmol. **Drag to rotate, scroll to zoom, right-drag to pan.**

> **Heads-up**:
> - First run downloads ~1.5 GB of model weights, then predicts in ~1–3 min per sequence.
> - Best for sequences ≤300 nt on Colab's free T4 GPU; longer may OOM.
> - For unstructured / homopolymer RNAs the predicted 3D will be a low-confidence "best guess"; the prediction is most meaningful for structured motifs (G-quadruplexes, kissing loops, riboswitches).
>
> For very long RNAs or highest fidelity, the [AlphaFold Server](https://alphafoldserver.com) is the gold standard (free, web, no install): paste your sequence there and download the CIF.

In [ ]:
#@title Predict and visualize 3D structure (Boltz-2) { display-mode: "form" }

#@markdown **Sequence** to predict (RNA only; A/C/G/U). Keep ≤300 nt for the free-tier GPU.
sequence = "GGGAGGGAGGGAGGGAA" #@param {type:"string"}

#@markdown ---

import os, sys, tempfile, pathlib, subprocess, shutil

# --- Install Boltz + py3Dmol (one-time, ~30 sec; weight download adds ~1.5 GB) ---
if shutil.which("boltz") is None:
    print("[3D] installing boltz + py3Dmol (one-time)")
    !{sys.executable} -m pip install -q boltz py3Dmol
try:
    import py3Dmol
except ImportError:
    !{sys.executable} -m pip install -q py3Dmol
    import py3Dmol

boltz_cmd = shutil.which("boltz")
if boltz_cmd is None:
    raise SystemExit("[3D] boltz CLI not found on PATH after install. Restart the runtime and re-run this cell.")

seq = sequence.upper().replace("T", "U").strip()
if not seq or any(c not in "ACGU" for c in seq):
    raise SystemExit(f"Sequence must contain only A/C/G/U. Got: {sequence[:50]!r}")
if len(seq) > 400:
    print(f"[3D] WARNING: sequence is {len(seq)} nt — may OOM on Colab's free T4. Trim to ≤300 nt if it fails.")

# --- Write Boltz YAML config ---
workdir = pathlib.Path(tempfile.mkdtemp(prefix="boltz_"))
config   = workdir / "input.yaml"
out_dir  = workdir / "out"
config.write_text(f"""version: 1
sequences:
  - rna:
      id: A
      sequence: {seq}
""")

# --- Predict (first run downloads weights; subsequent runs reuse cache) ---
print(f"[3D] predicting structure for {len(seq)}-nt RNA (~1–3 min) ...")
result = subprocess.run(
    [boltz_cmd, "predict", str(config), "--out_dir", str(out_dir), "--use_msa_server"],
)
if result.returncode != 0:
    raise SystemExit(f"[3D] boltz exited with status {result.returncode}. Scroll up for the error.")

# --- Find the structure output (Boltz writes CIF by default) ---
struct_files = sorted(out_dir.glob("**/*.cif")) + sorted(out_dir.glob("**/*.pdb"))
if not struct_files:
    raise SystemExit("[3D] no structure file found in boltz output — check the logs above")
struct_path = struct_files[0]
print(f"\n[3D] structure: {struct_path.name}")

# --- Render interactively with py3Dmol ---
fmt = "cif" if struct_path.suffix in (".cif", ".mmcif") else "pdb"
view = py3Dmol.view(width=720, height=520)
view.addModel(open(struct_path).read(), fmt)
view.setStyle({"cartoon": {"color": "spectrum"}})
view.setBackgroundColor("white")
view.zoomTo()
view.show()
print("[3D] drag to rotate · scroll to zoom · right-drag to pan")

## Tips & FAQ

**How do I read P(LLPS)?**
A value near `1.0` means the model is confident the RNA self-phase-separates (protein-free); near `0` means it expects no LLPS. The default threshold of `0.5` splits the `LLPS` / `no` call. Raise it to ~`0.6` if you want stricter "yes" calls (fewer false positives at the cost of some recall).

**Is a very high-scoring design actually reliable?**
Sometimes a design scores high mostly because of its base composition rather than its structure. Quick sanity check: take the design, **di-shuffle** it (preserves the dinucleotide composition but destroys higher-order structure), and re-score the shuffled copy. A truly structure-driven hit will lose most of its score; a composition artefact will keep it.

**Designs are predictions, not guarantees.**
The model proposes candidates it *believes* will phase-separate. Wet-lab validation is the deciding step — treat the FASTAs as a shortlist to test, not a finished answer.

**Where do the weights come from?**
Model weights are hosted on Hugging Face Hub: [quercuscode/rnaphaseek](https://huggingface.co/quercuscode/rnaphaseek). The Setup cell pulls them at runtime.

**Citation**
If you use RNAPhaseek in published work, please cite *Cheraghali et al.* (manuscript in preparation).

**Issues / questions**
Please open an issue on the [GitHub repository](https://github.com/QuercusCode/RNAPhaseek/issues).